In [1]:
!python  -V

Python 3.11.5


In [2]:
# Cell 1: Setup and Configuration
import json
import os
import sys
from pathlib import Path
from typing import List, Dict, Any
import hashlib

# Add project root to path
PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

from qdrant_client import QdrantClient, models
from qdrant_client.models import Distance, VectorParams, PointStruct
from fastembed import TextEmbedding

# Configuration
QDRANT_URL = "https://5ef7d200-3b5c-4874-8f95-e621d3d5d429.eu-central-1-0.aws.cloud.qdrant.io"
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY")
COLLECTION_NAME = "survival_strategies"
EMBEDDING_MODEL = "BAAI/bge-base-en-v1.5"
EMBEDDING_DIMENSION = 768


print("🎯 Configuration loaded for Survival Strategies RAG")

# Cell 2: Initialize Clients
def initialize_clients():
    """Initialize Qdrant client and embedding model"""
    # Initialize Qdrant Cloud client
    qdrant_client = QdrantClient(
        url=QDRANT_URL,
        api_key=QDRANT_API_KEY,
        timeout=60
    )
    
    # Initialize FastEmbed model
    embedding_model = TextEmbedding(model_name=EMBEDDING_MODEL)
    
    print("✅ Clients initialized successfully")
    return qdrant_client, embedding_model

🎯 Configuration loaded for Survival Strategies RAG


2025-10-19 02:19:54.017025536 [W:onnxruntime:Default, device_discovery.cc:164 DiscoverDevicesForPlatform] GPU device discovery failed: device_discovery.cc:89 ReadFileContents Failed to open file: "/sys/class/drm/card0/device/vendor"


In [3]:
qdrant_client, embedding_model = initialize_clients()

✅ Clients initialized successfully


In [4]:
test_queries = [
    "what if I wiggle to escape a quicksand, would I get out?",
    "what was the recorded damages to life and property in the wake of the world's largest hurricane or tornado?",
    "how do I know if water is safe to drink in the wilderness?",
    "what's the best way to signal for help if I'm lost?",
    "can I really use a watch as a compass like in the movies?",
    "how long can someone survive without water in the desert?",
    "what should I do if I encounter a mountain lion while hiking?",
    "is it true that you shouldn't eat snow if you're dehydrated?"
]

In [5]:
# Phase 1: Basic retrieval with realistic queries
def test_natural_queries():
    for query in test_queries:
        print(f"🔍 Query: '{query}'")
        print("-" * 80)
        
        # Get embedding
        query_embedding = list(embedding_model.embed([query]))[0]
        
        # Search Qdrant
        results = qdrant_client.query_points(
            collection_name=COLLECTION_NAME,
            query=query_embedding.tolist(),
            limit=3
        )
        
        # Display results
        for i, point in enumerate(results.points):
            print(f"{i+1}. Score: {point.score:.3f}")
            print(f"   Video: {point.payload['video_title']}")
            print(f"   Chapter: {point.payload['chapter_title']}")
            print(f"   Text: {point.payload['text'][:150]}...")
            print()
        
        print("=" * 80)
        print()

# Run the test
test_natural_queries()

🔍 Query: 'what if I wiggle to escape a quicksand, would I get out?'
--------------------------------------------------------------------------------
1. Score: 0.749
   Video: How to Survive Falling Into Quicksand
   Chapter: FALLING INTO QUICKSAND
   Text: You've probably seen people up to their neck in quicksand in the movies, but in real life, it's quite rare. If you happened to slip in, you'd likely j...

2. Score: 0.734
   Video: How To Escape Quicksand
   Chapter: How to Escape Quicksand
   Text: If you fall into quicksand, the last thing you want to do is struggle. Remember, quicksand is denser than water; the more you move, the more you sink....

3. Score: 0.732
   Video: How To Escape Quicksand
   Chapter: Stop Moving
   Text: Don't panic, but you need to get out fast. Your legs are in a vice, and the pressure on your cells can do some serious damage. If your legs are trappe...


🔍 Query: 'what was the recorded damages to life and property in the wake of the world's largest hur

In [6]:
def test_with_threshold(threshold=0.68):
    print(f"🔧 Testing with threshold: {threshold}")
    print("=" * 80)
    
    for query in test_queries:
        query_embedding = list(embedding_model.embed([query]))[0]
        results = qdrant_client.query_points(
            collection_name=COLLECTION_NAME,
            query=query_embedding.tolist(),
            limit=3
        )
        
        # Filter by threshold
        filtered_results = [r for r in results.points if r.score >= threshold]
        
        print(f"🔍 '{query}'")
        print(f"   Retrieved: {len(results.points)}, Above threshold: {len(filtered_results)}")
        
        if not filtered_results:
            print("   ❌ NO CONTEXT AVAILABLE")
        else:
            print(f"   ✅ Context available (best score: {filtered_results[0].score:.3f})")
        print()

# Test different thresholds
test_with_threshold(0.65)
test_with_threshold(0.68) 
test_with_threshold(0.70)

🔧 Testing with threshold: 0.65
🔍 'what if I wiggle to escape a quicksand, would I get out?'
   Retrieved: 3, Above threshold: 3
   ✅ Context available (best score: 0.749)

🔍 'what was the recorded damages to life and property in the wake of the world's largest hurricane or tornado?'
   Retrieved: 3, Above threshold: 3
   ✅ Context available (best score: 0.716)

🔍 'how do I know if water is safe to drink in the wilderness?'
   Retrieved: 3, Above threshold: 3
   ✅ Context available (best score: 0.764)

🔍 'what's the best way to signal for help if I'm lost?'
   Retrieved: 3, Above threshold: 3
   ✅ Context available (best score: 0.715)

🔍 'can I really use a watch as a compass like in the movies?'
   Retrieved: 3, Above threshold: 1
   ✅ Context available (best score: 0.657)

🔍 'how long can someone survive without water in the desert?'
   Retrieved: 3, Above threshold: 3
   ✅ Context available (best score: 0.788)

🔍 'what should I do if I encounter a mountain lion while hiking?'
   Retr

### retrieval

In [162]:
CONFIDENCE_THRESHOLD = 0.68

# Global conversation state
last_user_message = None
last_assistant_answer = None

def reset_conversation():
    global last_user_message, last_assistant_answer
    last_user_message = None
    last_assistant_answer = None

def update_conversation(user_msg, assistant_answer):
    global last_user_message, last_assistant_answer
    last_user_message = user_msg
    last_assistant_answer = assistant_answer

In [163]:
def determine_query_type(query, chunks_retrieved):
    """Determine query type based on conversation state and retrieval results"""
    has_history = (last_user_message is not None and 
                   last_assistant_answer is not None)
    
    if not has_history:
        return "NOT_IN_CONTEXT" if not chunks_retrieved else "FIRST_QUERY"
    else:
        return "NOT_IN_CONTEXT" if not chunks_retrieved else "FOLLOW_UP"

In [164]:
'''def retrieve_chunks(query, qdrant_client, embedding_model, top_k=5):
    """Retrieve chunks from Qdrant with confidence filtering"""
    query_embedding = list(embedding_model.embed([query]))[0]
    results = qdrant_client.query_points(
        collection_name="survival_strategies",
        query=query_embedding.tolist(),
        limit=top_k
    )
    
    # Filter by threshold
    filtered_results = [r for r in results.points if r.score >= CONFIDENCE_THRESHOLD]
    return filtered_results'''

def format_chunks_for_prompt(chunks):
    """Format retrieved chunks for LLM prompt"""
    if not chunks:
        return ""
    
    formatted_chunks = []
    for i, chunk in enumerate(chunks, 1):
        chunk_text = f"""
        Source {i}:
        Video: {chunk.payload['video_title']}
        Chapter: {chunk.payload['chapter_title']}  
        Content: {chunk.payload['text']}
        """
        formatted_chunks.append(chunk_text)
    
    return "\n".join(formatted_chunks)

In [165]:
def retrieve_chunks(query, qdrant_client, embedding_model, top_k=5):
    query_embedding = list(embedding_model.embed([query]))[0]
    results = qdrant_client.query_points(
        collection_name="survival_strategies",
        query=query_embedding.tolist(),
        limit=top_k
    )

    filtered_results = []
    for p in results.points:
        if p.score >= CONFIDENCE_THRESHOLD:
            filtered_results.append({
                "id": p.id,
                "text": p.payload.get("text", ""),
                "score": p.score,
                "payload": p.payload
            })
    return filtered_results

### llm augment

In [166]:
import time
from typing import List, Dict, Optional

NOT_IN_CONTEXT = "Not in context"
INFORMATION_NOT_FOUND_MSG = "I don't have information on that survival topic in my knowledge base."

def generate_response(
    user_query: str,
    context_chunks: List[Dict],   # each dict: {"id": "...", "text": "...", "score": ... (optional)}
    llm_client,
    model: str = "gpt-4o-mini",
    context_char_limit: int = 80000,
    temperature: float = 0.0,
):

    # 1) Sort/prepare chunks (simple ordering by provided score — not a neural re-ranker)
    chunks = list(context_chunks or [])
    #if chunks and isinstance(chunks[0], dict) and "score" in chunks[0]:
        # This is just ordering by the retrieval score (descending). True re-ranking is a separate model step.
        #chunks = sorted(chunks, key=lambda c: c.get("score", 0), reverse=True)

    # 2) Assemble context with provenance markers until char limit
    assembled = []
    length = 0
    for i, c in enumerate(chunks):
        cid = c.get("id", f"chunk_{i}")
        text = c.get("text", "") or ""
        block = f"[SOURCE: {cid}]\n{text}\n\n"
        if length + len(block) > context_char_limit:
            break
        assembled.append(block)
        length += len(block)

    context = "".join(assembled).strip()
    if not context:
        return INFORMATION_NOT_FOUND_MSG

    # 3) Strict system prompt (grounding)
    system_prompt = (
        "You are a Survival Guidance assistant that extracts ONLY factual information and procedural knowledge "
        "from the supplied Context. Use ONLY the text in Context. Do NOT use external knowledge or assumptions. "
        f"If the answer cannot be fully derived from the Context, respond EXACTLY with:{NOT_IN_CONTEXT}\n"
        "No speculation, no narration, preserve substantive instructions, provide structured outputs when asked."
    )

    # 4) Compose user prompt (question + context)
    user_prompt = (
        f"Question:\n{user_query}\n\n"
        f"Context (each block has provenance):\n\n{context}\n\n"
        "Task:\nUsing ONLY the Context above, answer the question. "
        f"If the required information is missing or not fully present, reply EXACTLY:{NOT_IN_CONTEXT}\n"
        "Be concise and factual with the required information for the user. No extra commentary."
    )

    # Debug prints (single set)
    print("SYSTEM PROMPT:\n", system_prompt)
    print("\nUSER PROMPT:\n", user_prompt)
    print("\n------------------ CALLING LLM ------------------\n")
    

    # 5) Call LLM (note: no max_tokens/timeout args)
    try:
        resp = llm_client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
            temperature=temperature,
        )
    except Exception:
        # If LLM call fails, prefer safe failure
        return INFORMATION_NOT_FOUND_MSG

    # 6) Extract model output (handle common shapes)
    raw = None
    if hasattr(resp, "choices"):
        try:
            raw = resp.choices[0].message.content
        except Exception:
            raw = None
    elif isinstance(resp, dict) and "choices" in resp:
        raw = resp["choices"][0]["message"]["content"]
    else:
        raw = str(resp)

    if raw is None:
        return INFORMATION_NOT_FOUND_MSG

    text = raw.strip()
    print(f"\n----- PROMPT SENT TO LLM -----\n{user_prompt}\n------------------------------\n")
    print(f"LLM RAW RESPONSE: {text}\n")

    # 7) Enforce exact failure token
    if text.strip().lower() == NOT_IN_CONTEXT.lower():
        return INFORMATION_NOT_FOUND_MSG

    #print(f"\n----- PROMPT SENT TO LLM -----\n{user_prompt}\n------------------------------\n")
    #print(f"LLM RAW RESPONSE: {text}\n")
    return text

In [167]:
def rewrite_query_for_qdrant(
    latest_user_message: str,
    last_llm_reply: str,
    current_user_question: str,
    llm_client,
    model: str = "gpt-4o-mini",
):

    system = (
        "You are a concise query rewriter for vector search in a RAG systm "
        "Given the assistant's last reply, the user's latest message, and the current user question "
        "(which previously returned no useful search results from the Knowledge Base), produce ONE single-sentence, standalone, "
        "human-language search query suitable for retrieving relevant chunks. "
        "Do NOT invent facts. If you cannot produce a safe rewrite without adding facts, return the current user question unchanged. "
    )

    user_prompt = (
        f"ASSISTANT_REPLY:\n{(last_llm_reply or '').strip()}\n\n"
        f"LATEST_USER_MESSAGE:\n{(latest_user_message or '').strip()}\n\n"
        f"CURRENT_USER_QUESTION (no/low-quality chunks):\n{(current_user_question or '').strip()}\n\n"
        "Return the question in only one sentence."
    )

    try:
        resp = llm_client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": system},
                {"role": "user", "content": user_prompt},
            ],
            temperature=0.0,
        )
    except Exception:
        return " ".join(current_user_question.split())

    # extract text (handle common response shapes)
    raw = None
    if hasattr(resp, "choices"):
        try:
            raw = resp.choices[0].message.content
        except Exception:
            raw = None
    elif isinstance(resp, dict) and "choices" in resp:
        raw = resp["choices"][0]["message"]["content"]
    else:
        raw = str(resp)

    if not raw:
        return " ".join(current_user_question.split())

    # keep first non-empty line, strip wrapping quotes, collapse whitespace
    for line in raw.splitlines():
        line = line.strip()
        if not line:
            continue
        if (line.startswith('"') and line.endswith('"')) or (line.startswith("'") and line.endswith("'")):
            line = line[1:-1].strip()
        final_rewritten_query = " ".join(line.split())
        print(f"📝 Rewritten query: {final_rewritten_query}")
        return final_rewritten_query

    final_rewritten_query = " ".join(current_user_question.split())
    print(f"Rewritten query:: {final_rewritten_query}")
    return final_rewritten_query

### rag control

In [168]:
# Configuration
CONFIDENCE_THRESHOLD = 0.68
COLLECTION_NAME = "survival_strategies"

# Initialize LLM client
llm_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# Global conversation state
last_user_message = None
last_assistant_answer = None

In [169]:
llm_client

In [170]:
def rag_pipeline(user_query):
    """Main RAG pipeline from user query to final answer (corrected wiring)."""

    # Step 1: Retrieve chunks for the raw user query
    chunks = retrieve_chunks(user_query, qdrant_client, embedding_model, top_k=5)
    chunks_retrieved = len(chunks) > 0

    # Step 2: Determine query type (uses global last_user_message/last_assistant_answer)
    query_type = determine_query_type(user_query, chunks_retrieved)

    # Step 3: Branch on query_type
    if query_type == "NOT_IN_CONTEXT":
        # No relevant chunks for a first query or follow-up -> safe failure & reset
        reset_conversation()
        return "I don't have information on that survival topic in my knowledge base."

    elif query_type == "FIRST_QUERY":
        # We have initial chunks; call generate_response with the chunks list (not formatted string).
        # generate_response expects: (user_query, context_chunks, llm_client, ...)
        response = generate_response(user_query, chunks, llm_client)
        update_conversation(user_query, response)
        return response

    elif query_type == "FOLLOW_UP":
        # We have conversation history: try to rewrite into a standalone retrieval query
        # Use the simple rewrite function signature:
        # rewrite_query_for_qdrant(latest_user_message, last_llm_reply, current_user_question, llm_client)
        rewritten_query = rewrite_query_for_qdrant(
            last_user_message,            # previous user message
            last_assistant_answer,        # previous assistant reply
            user_query,                   # current question that returned no/low chunks
            llm_client
        )

        # If rewrite fails it should return current_user_question per your rewrite function.
        # Now re-run retrieval with rewritten query
        new_chunks = retrieve_chunks(rewritten_query, qdrant_client, embedding_model, top_k=5)

        if not new_chunks:
            # Nothing found after rewriting -> safe failure and reset conversation
            reset_conversation()
            return "I don't have information on that survival topic in my knowledge base."

        # Otherwise, generate a grounded response from the retrieved chunks
        response = generate_response(rewritten_query, new_chunks, llm_client)
        # Update conversation with the user's original message and the assistant's response.
        # (Optionally you could also store rewritten_query somewhere for debugging.)
        update_conversation(user_query, response)
        return response

    else:
        # defensive fallback
        return "I don't have information on that survival topic in my knowledge base."


In [171]:
# Test function
def test_rag_pipeline():
  test_queries = ["what if I wiggle to escape a quicksand, would I get out?"]
     # "what was the recorded damages to life and property in the wake of the world's largest hurricane or tornado?",
     #"how do I know if water is safe to drink in the wilderness?",
     #"what's the best way to signal for help if I'm lost?",
    #"can I really use a watch as a compass like in the movies?",
    #"how long can someone survive without water in the desert?",
    #"what should I do if I encounter a mountain lion while hiking?",
    #"is it true that you shouldn't eat snow if you're dehydrated?"
    #]
    
  for query in test_queries:
    print(f"🧪 User: {query}")
    response = rag_pipeline(query)
    print(f"🤖 Assistant: {response}")
    print("-" * 50)

In [172]:
test_rag_pipeline()

🧪 User: what if I wiggle to escape a quicksand, would I get out?
SYSTEM PROMPT:
 You are a Survival Guidance assistant that extracts ONLY factual information and procedural knowledge from the supplied Context. Use ONLY the text in Context. Do NOT use external knowledge or assumptions. If the answer cannot be fully derived from the Context, respond EXACTLY with:Not in context
No speculation, no narration, preserve substantive instructions, provide structured outputs when asked.

USER PROMPT:
 Question:
what if I wiggle to escape a quicksand, would I get out?

Context (each block has provenance):

[SOURCE: 746894995501337991]
You've probably seen people up to their neck in quicksand in the movies, but in real life, it's quite rare. If you happened to slip in, you'd likely just sink halfway. Most of the time, quicksand isn't actually that deep. But it can still be incredibly dangerous. If you're trapped and there's no one around to help you, you could suffer hypothermia, dehydration or ev

In [173]:
# Test function
def test_followup_query_rewite():
  test_queries = ["Which water you are referring to and why would you loosen up water which is already loose?"]
     # "what was the recorded damages to life and property in the wake of the world's largest hurricane or tornado?",
     #"how do I know if water is safe to drink in the wilderness?",
     #"what's the best way to signal for help if I'm lost?",
    #"can I really use a watch as a compass like in the movies?",
    #"how long can someone survive without water in the desert?",
    #"what should I do if I encounter a mountain lion while hiking?",
    #"is it true that you shouldn't eat snow if you're dehydrated?"
    #]
    
  for query in test_queries:
    print(f"🧪 User follow up query: {query}")
    response = rag_pipeline(query)
    print(f"🤖 Assistant: {response}")
    print("-" * 50)

In [174]:
test_followup_query_rewite()

🧪 User follow up query: Which water you are referring to and why would you loosen up water which is already loose?
📝 Rewritten query: Which water are you referring to, and why would you need to loosen water that is already loose in quicksand?
SYSTEM PROMPT:
 You are a Survival Guidance assistant that extracts ONLY factual information and procedural knowledge from the supplied Context. Use ONLY the text in Context. Do NOT use external knowledge or assumptions. If the answer cannot be fully derived from the Context, respond EXACTLY with:Not in context
No speculation, no narration, preserve substantive instructions, provide structured outputs when asked.

USER PROMPT:
 Question:
Which water are you referring to, and why would you need to loosen water that is already loose in quicksand?

Context (each block has provenance):

[SOURCE: 3867695687837246310]
If you fall into quicksand, the last thing you want to do is struggle. Remember, quicksand is denser than water; the more you move, the m

In [175]:
# Test function
def test_rag_pipeline():
  test_queries = ["what was the recorded damages to life and property in the wake of the world's largest hurricane or tornado?"]
    
     #"how do I know if water is safe to drink in the wilderness?",
     #"what's the best way to signal for help if I'm lost?",
    #"can I really use a watch as a compass like in the movies?",
    #"how long can someone survive without water in the desert?",
    #"what should I do if I encounter a mountain lion while hiking?",
    #"is it true that you shouldn't eat snow if you're dehydrated?"
    #]
    
  for query in test_queries:
    print(f"🧪 User: {query}")
    response = rag_pipeline(query)
    print(f"🤖 Assistant: {response}")
    print("-" * 50)

In [179]:
test_rag_pipeline()

🧪 User: what was the recorded damages to life and property in the wake of the world's largest hurricane or tornado?
📝 Rewritten query: What were the recorded damages to life and property caused by the world's largest hurricane or tornado?
SYSTEM PROMPT:
 You are a Survival Guidance assistant that extracts ONLY factual information and procedural knowledge from the supplied Context. Use ONLY the text in Context. Do NOT use external knowledge or assumptions. If the answer cannot be fully derived from the Context, respond EXACTLY with:Not in context
No speculation, no narration, preserve substantive instructions, provide structured outputs when asked.

USER PROMPT:
 Question:
What were the recorded damages to life and property caused by the world's largest hurricane or tornado?

Context (each block has provenance):

[SOURCE: 14160575846262798516]
Today, we're comparing the scale of the largest tornadoes ever to carve their way across Earth, including one that caused nearly $1 billion in da

In [181]:
# Test function
def test_followup_query_rewite():
  test_queries = ["If I ever find my self near such things you just mentioned, what survival tips would you for me?"]
     # "what was the recorded damages to life and property in the wake of the world's largest hurricane or tornado?",
     #"how do I know if water is safe to drink in the wilderness?",
     #"what's the best way to signal for help if I'm lost?",
    #"can I really use a watch as a compass like in the movies?",
    #"how long can someone survive without water in the desert?",
    #"what should I do if I encounter a mountain lion while hiking?",
    #"is it true that you shouldn't eat snow if you're dehydrated?"
    #]
    
  for query in test_queries:
    print(f"🧪 User follow up query: {query}")
    response = rag_pipeline(query)
    print(f"🤖 Assistant: {response}")
    print("-" * 50)

In [182]:
test_followup_query_rewite()

🧪 User follow up query: If I ever find my self near such things you just mentioned, what survival tips would you for me?
📝 Rewritten query: What survival tips would you recommend if I find myself near a hurricane or tornado?
SYSTEM PROMPT:
 You are a Survival Guidance assistant that extracts ONLY factual information and procedural knowledge from the supplied Context. Use ONLY the text in Context. Do NOT use external knowledge or assumptions. If the answer cannot be fully derived from the Context, respond EXACTLY with:Not in context
No speculation, no narration, preserve substantive instructions, provide structured outputs when asked.

USER PROMPT:
 Question:
What survival tips would you recommend if I find myself near a hurricane or tornado?

Context (each block has provenance):

[SOURCE: 767258552528867108]
If you live in a trailer, mobile home or a boat, you must leave. These structures are neither heavy enough nor strong enough to withstand the kind of damage a hurricane can do. Go 

### ------FIN---------